# Cell 0

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from dateutil import parser
import os

clean_file = r"C:\Users\Admin\Desktop\Amazon\cleaned_amazon_data.csv"
output_dir = r"C:\Users\Admin\Desktop\Amazon\analysis_outputs"
os.makedirs(output_dir, exist_ok=True)

# Cell 1

In [ ]:
from dateutil import parser
import pandas as pd

def fix_date_cell(val):
    if pd.isna(val):
        return pd.NaT
    s = str(val).strip()
    s = s.replace("/", "-").replace("\\", "-").replace(".", "-")
    formats = ["%d-%m-%Y", "%d-%m-%y", "%Y-%m-%d", "%m-%d-%Y", "%d-%b-%Y", "%d-%B-%Y"]
    for fmt in formats:
        try:
            return pd.to_datetime(s, format=fmt, dayfirst=True, errors='raise')
        except Exception:
            pass
    try:
        return parser.parse(s, dayfirst=True, yearfirst=False)
    except Exception:
        return pd.NaT

# Cell 2

In [ ]:
df = pd.read_csv(clean_file, dtype=str)
print("Raw read shape:", df.shape)

df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_").str.replace("-", "_")
print("Columns:", df.columns.tolist())

for col in ["price","quantity","total_sales"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].str.replace(r"[^\d\.\-]", "", regex=True), errors="coerce")

if "date" in df.columns:
    df["date_parsed"] = df["date"].apply(fix_date_cell)
    print("Invalid dates:", df["date_parsed"].isna().sum())
    df = df.dropna(subset=["date_parsed"]).copy()
    df["date"] = df["date_parsed"]
else:
    print("No date column found")

# Cell 3

In [ ]:
print("After cleaning:", df.shape)
print(df.head(5).T)

# Cell 4

In [ ]:
numeric_cols = [c for c in df.columns if df[c].dtype.kind in "biufc"]
summary = df[numeric_cols].agg(['mean','median','min','max']).transpose().round(2)
summary

# Cell 5

In [ ]:
category_col = "category"
cat_agg = df.groupby(category_col).agg(
    price_mean=("price","mean"),
    price_median=("price","median"),
    units_sold_sum=("quantity","sum"),
    revenue_sum=("total_sales","sum"),
    orders=("order_id","nunique")
).reset_index().sort_values("revenue_sum", ascending=False)
cat_agg

# Chart 1

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(cat_agg[category_col], cat_agg["revenue_sum"])
plt.title("Total Revenue by Category")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Chart 2

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(cat_agg[category_col], cat_agg["units_sold_sum"])
plt.title("Units Sold by Category")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Cell 6

In [ ]:
corr = df[numeric_cols].corr()
corr

# Scatter

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(df["price"], df["total_sales"], alpha=0.6)
plt.xlabel("Price")
plt.ylabel("Total Sales")
plt.title("Price vs Total Sales")
plt.tight_layout()
plt.show()

# Cell 7

In [ ]:
top_by_revenue = df.sort_values("total_sales", ascending=False).head(20)
top_by_revenue

# Cell 8

In [ ]:
total_revenue = df["total_sales"].sum()
n_orders = df["order_id"].nunique()
aov = total_revenue / n_orders

threshold = np.percentile(df["total_sales"].dropna(), 90)
top_10_share = df[df["total_sales"] >= threshold]["total_sales"].sum() / total_revenue

metrics = {
    "total_revenue": total_revenue,
    "orders": n_orders,
    "AOV": round(aov,2),
    "Top10%RevenueShare": round(float(top_10_share),4)
}
metrics

# Cell 9

In [ ]:
df["year_month"] = df["date"].dt.to_period("M")
monthly = df.groupby("year_month")["total_sales"].sum().reset_index()
monthly["year_month"] = monthly["year_month"].dt.to_timestamp()
monthly